In [0]:
# =============================================================
# nb_audit_setup_smoke_test.py
# Layer   : Audit
# Purpose : Verify that the audit ADLS container is ready.
#
# Important:
#   This notebook cannot create an ADLS container from Spark.
#   Create the 'audit' container in the Azure Portal or Azure CLI first.
# =============================================================

from datetime import datetime, timezone


def _widget(name: str, default: str) -> str:
    try:
        value = dbutils.widgets.get(name).strip()
        if value:
            return value
    except Exception:
        pass
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name).strip()


STORAGE_ACCOUNT = _widget("storage_account", "petiot")
AUDIT_CONTAINER = _widget("audit_container", "audit")


def _run_notebook_with_fallback(label: str, paths: list[str]) -> None:
    last_error = None
    for path in paths:
        try:
            print(f"[loader] Loading {label}: {path}")
            get_ipython().run_line_magic("run", path)
            print(f"[loader] Loaded {label}: {path}")
            return
        except Exception as exc:
            last_error = exc
            print(f"[loader] Could not load {path}: {str(exc)[:250]}")
    raise RuntimeError(f"Unable to load {label}. Last error: {last_error}")


_run_notebook_with_fallback(
    "set_auth_context",
    [
        "/Workspace/Shared/pet-analytics/shared/set_auth_context.py.ipynb",
        "/Workspace/Shared/pet-analytics/shared/set_auth_context",
        "/Shared/pet-analytics/shared/set_auth_context",
    ],
)

_run_notebook_with_fallback(
    "audit_config",
    [
        "/Workspace/Shared/pet-analytics/audit/audit_config",
        "/Workspace/Shared/pet-analytics/audit/audit_config.py",
        "/Shared/pet-analytics/audit/audit_config",
    ],
)

_run_notebook_with_fallback(
    "audit_writer",
    [
        "/Workspace/Shared/pet-analytics/audit/audit_writer",
        "/Workspace/Shared/pet-analytics/audit/audit_writer.py",
        "/Shared/pet-analytics/audit/audit_writer",
    ],
)
ensure_audit_container_access.__globals__.update({'audit_path': audit_path, 'dbutils': dbutils, 'spark': spark})


print("=" * 80)
print("  Audit container smoke test")
print(f"  storage_account : {STORAGE_ACCOUNT}")
print(f"  audit_container : {AUDIT_CONTAINER}")
print("=" * 80)

ensure_audit_container_access(STORAGE_ACCOUNT, AUDIT_CONTAINER)

smoke_record = [{
    "test_name": "audit_container_write_smoke_test",
    "status": "PASS",
    "storage_account": STORAGE_ACCOUNT,
    "audit_container": AUDIT_CONTAINER,
    "tested_at_utc": datetime.now(timezone.utc).isoformat(),
    "message": "Audit container is accessible and Delta append is working.",
}]

_append_records(smoke_record, "_smoke_test", STORAGE_ACCOUNT, AUDIT_CONTAINER)

print("Smoke test complete. You can now run nb_run_fact_validation.py.")
display(read_audit_table("_smoke_test", STORAGE_ACCOUNT, AUDIT_CONTAINER).orderBy("_audit_write_ts", ascending=False))